In [5]:
import pyterrier as pt
import pandas as pd
import ir_datasets
import torch
from tqdm import tqdm
from transformers import DefaultDataCollator
from itertools import islice
import pandas as pd
import numpy as np
import re
import string

def flatten(xss):
    return [x for xs in xss for x in xs]

class OHSUMEDProportionateRandomSensitiveDataset():
    
    def __init__(self):
        print("Loading docs")
        
        ohsumed_training_docs = self.get_d2qmm()
        
        sensitive_docs = ohsumed_training_docs[ohsumed_training_docs["sensitivity"] == 1].reset_index()
        non_sensitive_docs = ohsumed_training_docs[ohsumed_training_docs["sensitivity"] == 0].head(len(sensitive_docs)).reset_index()
        self.len = len(ohsumed_training_docs)

        self.all_docs = ohsumed_training_docs
        self.sensitive_docs = sensitive_docs
        self.non_sensitive_docs = non_sensitive_docs
        self.queries = {index: f"{row['querygen']}" for index, row in ohsumed_training_docs.iterrows()}
        print("Docs loaded")

    def get_d2qmm(self):
        d2qmm = pd.read_pickle("/nfs/primary/sas_cross_encoder/d2qmm_ohsumed_10samples_0.1filter.pkl")
        max_score_indices = []
        for row_scores in d2qmm['querygen_score']:
            if len(row_scores) > 0:
                max_score_indices.append(np.argmax(np.array(row_scores)))
            else:
                # Handle the case of an empty list
                max_score_indices.append(None)
    
        # Update 'querygen' to keep only the query with the top score
        d2qmm['querygen'] = [queries.split('\n')[index] if (index is not None and queries) else None for queries, index in zip(d2qmm['querygen'], max_score_indices)]
    
        # Drop the 'querygen_score' column as it's no longer needed
        d2qmm = d2qmm.drop('querygen_score', axis=1)
        d2qmm['querygen'] = d2qmm['querygen'].str.replace(f"[{re.escape(string.punctuation)}]", "", regex = True)
        
        return d2qmm
        
    def __len__(self):
        return self.len
        
    def __getitem__(self, idx):
        current_doc = self.all_docs.iloc[[idx]]
        query = self.queries[idx]

        pos_doc = current_doc["text"].iloc[0]        
        if current_doc.sensitivity.iloc[0] == 1:
            neg_doc = self.sensitive_docs['text'].sample(n = 1, random_state = 42).iloc[0]
        else:
            neg_doc = self.all_docs['text'].sample(n = 1, random_state = 42).iloc[0]
            
        docs = [query, pos_doc, neg_doc]
        scores = torch.tensor([0, 0, 0]).view(1, -1)
        return docs, scores

dataset = OHSUMEDDataset()
dataset.__getitem__(0)

Loading docs
Docs loaded


(['what is acetaldehyde binding to',
  'The binding of acetaldehyde to the active site of ribonuclease: alterations in catalytic activity and effects of phosphate. Ribonuclease A was reacted with [1-13C,1,2-14C]acetaldehyde and sodium cyanoborohydride in the presence or absence of 0.2 M phosphate. After several hours of incubation at 4 degrees C (pH 7.4) stable acetaldehyde-RNase adducts were formed, and the extent of their formation was similar regardless of the presence of phosphate. Although the total amount of covalent binding was comparable in the absence or presence of phosphate, this active site ligand prevented the inhibition of enzymatic activity seen in its absence. This protective action of phosphate diminished with progressive ethylation of RNase, indicating that the reversible association of phosphate with the active site lysyl residue was overcome by the irreversible process of reductive ethylation. Modified RNase was analysed using 13C proton decoupled NMR spectroscopy. 

In [7]:
from tqdm import tqdm

for i in tqdm(range(334134), total = 334134):
    dataset.__getitem__(i)

  0%|          | 1650/334134 [00:06<21:54, 252.97it/s]


KeyboardInterrupt: 